# Training a Computer-Using Agent (CUA) for GUI Task Automation
#### Group 5 — CS 1090B

**Members:** William Gao, Jamie Linsdell, Tim Zhang, Zaina Azhar, Guy Mahieu

---

#### Problem Statement

Given a screenshot and a natural language referring expression, predict the `(x, y)` coordinate of the described UI element — enabling computer-using agents to interact with GUIs without hard-coded selectors.

#### Research Question

Can a sub-1B vision-language model (Florence-2, 0.77B), LoRA fine-tuned on a subset of UGround-V1-Data, match or approach a 2B baseline (Qwen2-VL) and a 7B specialist (UGround-V1) on UI element grounding accuracy?

#### MS3 Recap

MS3 established zero-shot baselines for both Florence-2 and Qwen2-VL on 500 samples of UGround-V1-Data. Key results:
- Florence-2 zero-shot: ~70% prediction rate (bounding box → center point)
- Qwen2-VL zero-shot: direct `(x, y)` output, stronger baseline
- Gap motivates fine-tuning Florence-2 with task-specific supervision


## Table of Contents

- [0. Setup](#0-setup)
- [1. Data Loading](#1-data-loading)
- [2. MS3 Baseline Results (Reference)](#2-ms3-baseline-results)
- [3. Florence-2 Fine-Tuning](#3-florence-2-fine-tuning)
  - [3.1 Loss Function Overview](#31-loss-function-overview)
  - [3.2 Setup & LoRA Config](#32-setup--lora-config)
  - [3.3 Training Data](#33-training-data)
  - [3.4 Smoke Test](#34-smoke-test)
  - [3.5 Ablation A — CE Only](#35-ablation-a--ce-only)
  - [3.6 Ablation B — CE + L2](#36-ablation-b--ce--l2)
  - [3.7 Ablation C — CE + Huber](#37-ablation-c--ce--huber)
  - [3.8 Ablation D — CE + Knowledge Distillation (UGround-V1-2B)](#38-ablation-d--ce--knowledge-distillation)
  - [3.9 Results Comparison](#39-results-comparison)
- [4. Analysis & Breakdown](#4-analysis--breakdown)
  - [4.1 By Element Type](#41-by-element-type)
  - [4.2 By Spatial Region](#42-by-spatial-region)
- [5. Qwen2-VL Fine-Tuning (Tim)](#5-qwen2-vl-fine-tuning)


## 0. Setup

> **Note on `transformers` versioning:**
> Florence-2 requires `transformers==4.44.2`. Qwen2-VL requires `>=4.45.0`.
> Run Florence sections first, then the upgrade cell before Qwen sections.
> See the "How to Run" guide in the MS3 notebook for full details.


In [ ]:
# Install dependencies (Florence-2 compatible)
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.44.2", "accelerate", "peft",
                "qwen-vl-utils", "einops"], check=True)
print("Dependencies installed.")


In [ ]:
import os, re, json, io, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from datasets import load_dataset
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


## 1. Data Loading

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# Load parquet files from Drive
import glob as _glob

EXTRACT_DIR = "/content/drive/MyDrive/Too CUA for School/uground_subset"
_pq = [f for f in _glob.glob(f"{EXTRACT_DIR}/**/*.parquet", recursive=True)
       if os.path.getsize(f) > 0]
print(f"Found {len(_pq)} parquet files")

ds = load_dataset("parquet", data_files=_pq, split="train")
print(f"Dataset: {len(ds):,} rows")


## 2. MS3 Baseline Results (Reference)

Zero-shot results from MS3 notebook on 500 UGround samples.
These are the benchmarks all fine-tuned models are compared against.


In [ ]:
# MS3 zero-shot results (paste in from MS3 notebook output)
# Update these numbers after running MS3 notebook.

ms3_results = {
    "Florence-2 zero-shot": {
        "n_samples": 500,
        "parse_rate": None,       # fill in from MS3 output
        "acc_50px": None,
        "acc_100px": None,
        "mean_err_px": None,
    },
    "Qwen2-VL-2B zero-shot": {
        "n_samples": 500,
        "parse_rate": None,       # fill in from MS3 output
        "acc_50px": None,
        "acc_100px": None,
        "mean_err_px": None,
    },
}

print("MS3 baseline results loaded (fill in from MS3 notebook output).")


## 3. Florence-2 Fine-Tuning

### 3.1 Loss Function Overview

We ablate four loss configurations on the same LoRA fine-tuning setup:

| Ablation | Loss | Description |
|----------|------|-------------|
| A | CE only | Standard next-token cross-entropy on `<loc_X><loc_Y>` tokens |
| B | CE + L2 | Adds soft L2 penalty on predicted coordinate expectation |
| C | CE + Huber | Replaces L2 with Huber (Smooth L1) — less sensitive to outliers |
| D | CE + KD | Adds distillation term from UGround-V1-2B teacher predictions |

All variants share the same LoRA config, training data, and number of steps
so results are directly comparable.

**Why these four:**
- CE only establishes the LM baseline with no spatial inductive bias
- L2 vs Huber tests sensitivity to outlier predictions
- KD tests whether a domain-specialist teacher (trained on this exact dataset) can close the gap to Qwen2-VL


### 3.2 Setup & LoRA Config

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

FLORENCE_MODEL_ID = "multimodalart/Florence-2-large-no-flash-attn"

print(f"Loading {FLORENCE_MODEL_ID}...")
florence_processor = AutoProcessor.from_pretrained(
    FLORENCE_MODEL_ID, trust_remote_code=True)
florence_model = AutoModelForCausalLM.from_pretrained(
    FLORENCE_MODEL_ID, torch_dtype=torch.float16, trust_remote_code=True
).to(DEVICE)

# LoRA config — target the language model decoder only
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
florence_model = get_peft_model(florence_model, lora_config)
florence_model.print_trainable_parameters()


### 3.3 Training Data

In [ ]:
# Build train/eval split — disjoint from MS3 eval set
# MS3 used a fixed random seed (123) for eval; we use a different seed here.

TRAIN_N = 1500   # training items
EVAL_N  = 500    # held-out eval (same size as MS3 for comparability)
random.seed(42)

all_indices = list(range(len(ds)))
random.shuffle(all_indices)
train_indices = all_indices[:TRAIN_N]
eval_indices  = all_indices[TRAIN_N:TRAIN_N + EVAL_N]

print(f"Train: {len(train_indices):,} items")
print(f"Eval : {len(eval_indices):,} items")
print(f"Overlap: {len(set(train_indices) & set(eval_indices))} (should be 0)")


### 3.4 Smoke Test

In [ ]:
# Run a quick smoke test (20 steps) before committing to a full training run.
# Confirms: loc token vocab, tensor shapes, loss components, sample inference.
# Should complete in ~3-5 min on A100.

SMOKE = True   # set to False to skip

if SMOKE:
    print("Running smoke test (20 steps)...")
    # TODO: wire to florence_finetune.py smoke test function
    print("Smoke test complete.")


### 3.5 Ablation A — CE Only

In [ ]:
# Train with cross-entropy loss only (no spatial term).
# LOSS_CONFIG = "ce_only"
# TODO: run training


### 3.6 Ablation B — CE + L2

In [ ]:
# Train with CE + L2 expectation loss (from MS3 scaffold).
# LOSS_CONFIG = "ce_l2"
# TODO: run training


### 3.7 Ablation C — CE + Huber

In [ ]:
# Train with CE + Huber (Smooth L1) loss.
# Huber = L2 for small errors, L1 for large errors — less sensitive to outliers.
# delta = 50 (in normalized 0-999 space)
# LOSS_CONFIG = "ce_huber"
# TODO: run training


### 3.8 Ablation D — CE + Knowledge Distillation

**Teacher model:** `osunlp/UGround-V1-2B` — trained on UGround-V1-Data (our exact dataset).
Outputs `(x, y)` in [0, 999] space, same format as our predictions.

$$\mathcal{L} = \mathrm{CE}(\hat{y}, y_{GT}) + \lambda \cdot L2(\hat{y}, y_{\text{UGround}})$$

The distillation term pulls Florence toward UGround's predictions, which are
domain-specialist outputs from a 2B model trained on this dataset.


In [ ]:
# Step 1: Generate UGround teacher labels on training set (run once, save to Drive)
TEACHER_LABELS_PATH = "/content/drive/MyDrive/Too CUA for School/uground_teacher_labels.json"

if not os.path.exists(TEACHER_LABELS_PATH):
    print("Generating UGround-V1-2B teacher labels...")

    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.45.0", "qwen-vl-utils"], check=True)

    from transformers import Qwen2VLForConditionalGeneration, AutoProcessor as AP2
    from qwen_vl_utils import process_vision_info

    TEACHER_ID = "osunlp/UGround-V1-2B"
    print(f"Loading teacher: {TEACHER_ID}")
    teacher_proc  = AP2.from_pretrained(TEACHER_ID)
    teacher_model = Qwen2VLForConditionalGeneration.from_pretrained(
        TEACHER_ID, torch_dtype=torch.bfloat16, device_map="auto")
    teacher_model.eval()

    # TODO: run inference on train_indices, save (x, y) predictions
    print("Teacher labels saved to Drive.")
else:
    print(f"Teacher labels already exist at {TEACHER_LABELS_PATH}")


In [ ]:
# Step 2: Train Florence with CE + KD loss
# LOSS_CONFIG = "ce_kd"
# TODO: run training with teacher labels


### 3.9 Results Comparison

In [ ]:
# Fill in after all training runs complete.
# Rows: zero-shot baselines + 4 fine-tuned variants.

results_table = pd.DataFrame([
    {"Model": "Florence-2 zero-shot (MS3)",   "Acc@50px": None, "Acc@100px": None, "Mean err (px)": None},
    {"Model": "Qwen2-VL-2B zero-shot (MS3)",  "Acc@50px": None, "Acc@100px": None, "Mean err (px)": None},
    {"Model": "Florence-2 FT — CE only",      "Acc@50px": None, "Acc@100px": None, "Mean err (px)": None},
    {"Model": "Florence-2 FT — CE + L2",      "Acc@50px": None, "Acc@100px": None, "Mean err (px)": None},
    {"Model": "Florence-2 FT — CE + Huber",   "Acc@50px": None, "Acc@100px": None, "Mean err (px)": None},
    {"Model": "Florence-2 FT — CE + KD",      "Acc@50px": None, "Acc@100px": None, "Mean err (px)": None},
])
print(results_table.to_string(index=False))


## 4. Analysis & Breakdown

### 4.1 By Element Type

Breaks down accuracy by UI element type (button, input, link, icon, etc.)
to address TF feedback: *"analysis is not broken down by key factors identified in EDA."*


In [ ]:
# TODO: after eval runs, group results by element type
# Element type is parsed from the referring expression (same as MS3 EDA)
# Plot: grouped bar chart of Acc@100px per element type, per model variant


### 4.2 By Spatial Region

Breaks down accuracy by screen quadrant (top-left, top-right, bottom-left, bottom-right)
to connect EDA spatial bias finding to model performance.

TF feedback: *"dataset bias (top-heavy spatial distribution) influences results"*


In [ ]:
# TODO: after eval runs, group results by quadrant of GT (x, y)
# Quadrant: x < 500 vs x >= 500, y < 500 vs y >= 500 (in 0-999 space)
# Plot: heatmap or grouped bars of accuracy per quadrant


## 5. Qwen2-VL Fine-Tuning (Tim)

> **Note:** Requires `transformers>=4.45.0`. Run the upgrade cell and restart
> the runtime before running cells in this section.

LoRA fine-tuning of Qwen2-VL-2B on the same training split used for Florence-2.
Results feed into the Section 3.9 comparison table.


In [ ]:
# Upgrade transformers for Qwen2-VL
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.45.0"],
               check=True)
print("Restart runtime now, then run Qwen cells below.")
